## Initial BLASTp Search

In [ ]:
import os
import time
import subprocess
import pandas as pd

pep_folder = 'data/1k-species-data/y1000p_pep_files'
db_folder = 'data/1k-species-data/blast_dbs'
hits_folder = 'data/1k-species-data/CHC1_hits'
master_fasta = 'data/1k-species-data/1k-CHC1.fasta'
metadata_csv = 'data/1k-species-data/CHC1_metadata.csv'
query_fasta = 'data/1k-species-data/CHC1_seed.fasta'

'''
Add fuzzy-string logic (fuzzy wuzzy, other tool) to account for human error.
'''

metadata = []
species_found = []

for pep_file in os.listdir(pep_folder):
    if not pep_file.endswith('.final.pep'):
        continue
    
    if pep_file.startswith('yH'):
        species_name = pep_file.split('_', 1)[1].rsplit('_', 1)[0]
        for other_file in os.listdir(pep_folder):
            if species_name in other_file and 'final' in other_file and not other_file.startswith('yH'):
                pep_file = other_file
    else:
        species_name = pep_file.rsplit('.', 2)[0]
                
    print(pep_file)
    print(species_name)

    if species_name in species_found:
        continue
    species_found.append(species_name)
    print(f'Processing {species_name}')
    pep_path = os.path.join(pep_folder, pep_file)

    species_folder = os.path.join(db_folder, species_name)
    os.makedirs(species_folder, exist_ok=True)
    db_path = os.path.join(species_folder, species_name)
    subprocess.run([
        'makeblastdb',
        '-in', pep_path,
        '-dbtype', 'prot',
        '-out', db_path
    ], check=True)
    
    hits_file = os.path.join(hits_folder, f"{species_name}_CHC1_hits.txt")
    subprocess.run([
        'blastp',
        '-query', query_fasta,
        '-db', db_path,
        '-out', hits_file,
        '-outfmt', '6'
    ], check=True)

    time.sleep(0.1)
    
    if os.path.exists(hits_file):
        with open(hits_file) as f:
            lines = f.readlines()
        if lines:
            # BLAST outfmt 6: qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore
            top_hit = max(lines, key=lambda x: float(x.split()[2]))  # highest % identity
            cols = top_hit.strip().split()
            sseqid, pident, evalue, bitscore = cols[1], float(cols[2]), float(cols[10]), float(cols[11])
            
            seq_lines = []
            record = False
            with open(pep_path) as pep_f:
                for line in pep_f:
                    if line.startswith('>'):
                        record = sseqid in line
                        continue
                    if record:
                        seq_lines.append(line.strip())
            seq = '\n'.join([line.strip().rstrip('*') for line in seq_lines])
            
            with open(master_fasta, 'a') as f_out:
                f_out.write(f'>{species_name}|{sseqid}\n{seq}\n')
            
            metadata.append([species_name, sseqid, len(seq.replace('\n','')), pident, evalue, bitscore])
        else:
            print(f'No hits found for {species_name}')

df = pd.DataFrame(metadata, columns=['Species', 'Hit_ID', 'Seq_Length', 'Percent_Identity', 'Evalue', 'Bitscore'])
df.to_csv(metadata_csv, index=False)

## BLAST Search with Saccharomyces Cerevisiae CHC1 over Orthogroup FASTAs

In [95]:
local_path = '/Users/georgecrawford/Documents/yeast-clathrin-conservation/'

In [71]:
import os
from pathlib import Path

orthogroup_dir = Path(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences').expanduser()
os.chdir(orthogroup_dir)
print('Current folder:', os.getcwd())

Current folder: /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/y1000p_orthofinder/Orthogroup_Sequences


In [72]:
all_fasta_file = 'all_orthogroups.fa'

fasta_files = list(orthogroup_dir.glob('*.fa')) + list(orthogroup_dir.glob('*.fasta'))
print(f"Found {len(fasta_files)} FASTA files.")

with open(all_fasta_file, 'w') as outfile:
    for fasta in fasta_files:
        with open(fasta, 'r') as infile:
            outfile.write(infile.read())

print(f"All orthogroups written to {all_fasta_file}")

Found 72380 FASTA files.
All orthogroups written to all_orthogroups.fa


In [73]:
!makeblastdb -in all_orthogroups.fa -dbtype prot



Building a new DB, current time: 11/04/2025 15:01:50
New DB name:   /Users/georgecrawford/Documents/yeast-clathrin-conservation/genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/all_orthogroups.fa
New DB title:  all_orthogroups.fa
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 6971672 sequences in 58.8001 seconds.




In [76]:
ss_chc1_file = local_path + "genome_analyses/blast_results/s_cerevisiae_CHC1.fasta"
blast_output_file = local_path + "genome_analyses/blast_results/SS_CHC1_vs_orthogroups.txt"

!blastp -query {ss_chc1_file} -db all_orthogroups.fa -out {blast_output_file} -outfmt 6 -max_target_seqs 5

In [78]:
with open(blast_output_file) as f:
    lines = f.readlines()
    print('Top BLAST hits:')
    for line in lines[:10]:
        print(line.strip())

Top BLAST hits:
saccharomyces_cerevisiae_YGL206C	YGL206C|saccharomyces_cerevisiae.sgd	100.000	1653	0	0	1	1653	1	1653	0.0	3387
saccharomyces_cerevisiae_YGL206C	g004661.m1|saccharomyces_cerevisiae.final	100.000	1653	0	0	1	1653	1	1653	0.0	3387
saccharomyces_cerevisiae_YGL206C	g004575.m1|saccharomyces_paradoxus.final	98.246	1653	29	0	1	1653	1	1653	0.0	3336
saccharomyces_cerevisiae_YGL206C	g004716.m1|saccharomyces_mikatae.final	96.854	1653	52	0	1	1653	1	1653	0.0	3300
saccharomyces_cerevisiae_YGL206C	g004535.m1|saccharomyces_arboricola.final	96.310	1653	61	0	1	1653	1	1653	0.0	3281


### Top sequence's ID (YGL206C) found in orthogroup with ID 0000944 

#### Next steps:
- Determine actually how many species are in this orthogroup (total number of sequences in it is 1307, but this is because many of them have two annotations -- one internal (with the format g######.m1) and one universal (like Saccharomyces cerevisiae's YGL206C). The total number needs to be less anyway (1154).
- Run a reciprocal BLAST of orthogroup's sequences to S. cerevisiae's proteome. This will fully confirm that these are CHC1s.
- Deal with the multiple abnormally short sequences present in this orthogroup FASTA file. The output below shows the sequence lengths.

In [96]:
from Bio import SeqIO

CHC_OG_records = list(SeqIO.parse(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/OG0000944.fasta', 'fasta'))

print([len(seq_record.seq) for seq_record in CHC_OG_records])

[1681, 1680, 1686, 1675, 1665, 1702, 1690, 1678, 1666, 1704, 1676, 1685, 1682, 1682, 1674, 1675, 1649, 1676, 1681, 1670, 1660, 1656, 1647, 1665, 1671, 1643, 1635, 1635, 1669, 1673, 1673, 1643, 865, 727, 1653, 1651, 1649, 1651, 1683, 1674, 1653, 1643, 1664, 1653, 1654, 1654, 1654, 1656, 1656, 1658, 1657, 1673, 1658, 1665, 1665, 1665, 1665, 1665, 1665, 1662, 1660, 1678, 1664, 1526, 121, 1653, 1653, 1653, 1652, 1631, 1653, 1653, 1653, 1653, 1670, 1673, 1673, 1672, 2889, 1668, 1668, 1666, 1681, 1672, 1666, 1673, 330, 399, 789, 1673, 1673, 1604, 1662, 1669, 1655, 1655, 1655, 283, 95, 1472, 512, 1655, 1653, 1655, 1653, 1653, 1653, 1654, 1655, 1655, 1654, 1655, 1650, 1637, 1629, 1637, 1650, 1654, 1655, 995, 652, 1654, 1654, 1655, 1655, 1654, 1653, 1654, 1626, 1655, 182, 1655, 1659, 1648, 1654, 1652, 1664, 1654, 1648, 1654, 1651, 1652, 1661, 1670, 1659, 1720, 1667, 1658, 1653, 166, 1656, 1657, 1653, 1652, 1663, 1667, 1668, 1670, 1648, 1671, 1657, 1659, 1658, 1670, 1654, 1653, 1661, 1665, 1662,

### S. Cerevisiae CLC1 (YGR167W) found in orthogroup with ID 0002385

Below are the lengths of the sequences in this orthogroup (need to do further analysis, but from first glance it definitely seems like most/all are full sequences)

There are also a total of 1208 sequences in this orthogroup, which is more than the needed 1154. Again, this is likely due to certain sequences being repeated due to separate annotations. Regardless, next steps include:
- Finding how many unique sequences there are in this orthogroup

In [100]:
CLC_OG_records = list(SeqIO.parse(local_path + 'genome_analyses/y1000p_orthofinder/Orthogroup_Sequences/OG0002385.fasta', 'fasta'))

print([len(seq_record.seq) for seq_record in CLC_OG_records])

[253, 242, 290, 319, 281, 301, 277, 266, 196, 260, 236, 246, 291, 221, 241, 228, 157, 240, 233, 233, 225, 225, 236, 218, 225, 217, 222, 222, 222, 222, 229, 232, 209, 149, 155, 157, 146, 239, 236, 226, 215, 249, 201, 235, 229, 241, 217, 217, 217, 216, 238, 213, 212, 213, 213, 213, 213, 213, 204, 225, 244, 249, 134, 236, 233, 233, 234, 236, 236, 236, 234, 234, 230, 238, 239, 237, 212, 217, 229, 232, 218, 228, 217, 218, 218, 229, 253, 221, 270, 235, 238, 238, 235, 236, 201, 217, 204, 205, 216, 219, 216, 194, 199, 217, 220, 200, 216, 215, 223, 211, 212, 201, 192, 120, 216, 217, 229, 220, 212, 199, 216, 229, 221, 760, 238, 125, 203, 212, 217, 229, 125, 215, 208, 156, 228, 230, 201, 215, 212, 210, 218, 218, 240, 177, 200, 220, 212, 197, 125, 218, 222, 226, 221, 217, 188, 230, 119, 218, 227, 220, 222, 213, 209, 199, 215, 243, 222, 227, 202, 216, 202, 230, 213, 211, 175, 219, 207, 217, 206, 220, 213, 235, 209, 234, 236, 192, 204, 202, 202, 213, 221, 218, 212, 236, 221, 210, 224, 242, 230, 226,

## Emboss Cons Algorithm Recreation (tested on 2 CHC1s in the Sporopachydermia clade)

In [56]:
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Align import substitution_matrices
from Bio.Align.AlignInfo import SummaryInfo

alignment = AlignIO.read('data/CHC1-fastas-by-clade/Sporopachydermia-clade_aligned_CHCs.fasta', 'fasta')

print(f'Number of sequences: {len(alignment)}')
print(f'Alignment length: {alignment.get_alignment_length()}')

seq_weight = 1 # Enable different weights for different sequences
blosum62 = substitution_matrices.load('BLOSUM62')
plurality_value = 0.5 * (len(alignment) * seq_weight)
consensus_seq = ''

for i in range(alignment.get_alignment_length()):
    site = alignment[:, i]
    
    if all(aa == '-' for aa in site):
        consensus_seq += '-'
        continue
    elif all(aa == site[0] for aa in site):
        consensus_seq += site[0]
        continue
        
    scores = {}
    for k, residue in enumerate(site):
        if residue == '-':
            continue
        scores[residue] = [0, 0]
        for j, other_residue in enumerate(site):
            if k == j or other_residue == '-':
                continue
            scores[residue][0] += blosum62[residue, other_residue] * seq_weight    
            scores[residue][1] += seq_weight if blosum62[residue, other_residue] > 0 else 0 # Fix

    best_residue = max(scores, key=lambda r: scores[r][0])
    highest_score = scores[best_residue][0]
    positive_matches = scores[best_residue][1]
    
    if positive_matches >= plurality_value:
        consensus_seq += best_residue
    else:
        consensus_seq += 'x'

Number of sequences: 2
Alignment length: 4194


In [57]:
print(consensus_seq)

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------